# Question 1 (15 points)
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of an **inequality form polyhedron**: $P = \{x \in R^n: Ax >= b\}$
* Assume that `u` is an arbitrary n-dimensional vector
* The function must return `true` if `u` is a degenerate basic solution of $P$ and `false` otherwise
* Keep in mind that `u` may or may not be a basic solution of $P$. If it is not a basic solution, the function must return `false`

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `A` with `A::Matrix`, `b` with `b::Vector` or `u` with `u::Vector`).
* Make sure that the function always returns either `true` or `false` irrespective of the input.

## Hints
* This question is very similar to Question 3 of HW 3
* Test your function on the examples from Lecture 10 to verify if you have implemented it correctly (you can add other tests as well)

In [1]:
using LinearAlgebra

function is_degenerate_basic_solution(A, b, u)
    m = size(A, 1)
    n = size(A, 2)
    
    # check if inputs are consistent
    if length(b) != m || length(u) != n
        @warn("Input data is inconsistent!")
        return false
    end
    
    #
    active_set = []
    for i in 1:m
        if dot(A[i,:] , u) ≈ b[i]
            [i]
            push!(active_set , i)
        end
    end
    
    if length(active_set) >  n
        return true
    else 
        return false
    end
    #
    
end

is_degenerate_basic_solution (generic function with 1 method)

In [2]:
A = [-1.0 -2.0;
     -2.0 -1.0;
     -1.0  0.0;
      1.0 0.0;
      0.0  1.0]
b = [-3.0, -3.0, -1.5, 0.0, 0.0]
u = [1.5, 0]

is_degenerate_basic_solution(A, b, u)

true

# Question 2 (20 points)
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of a **standard form polyhedron**: $P = \{x \in R^n: Ax = b, x >= 0\}$
* Assume that `basic_col_indices` is an array of $m$ numbers which correspond to $m$ column indices of `A`
    - For example, if $n=5$ and $m=3$, then `basic_col_indices = [1,4,5]` would correspond to $B(1) = 1, B(2) = 4, B(3) = 5$ in the lecture notes
* The function must return `true` if the basis matrix corresponding to `basic_col_indices` defines a basic feasible solution of $P$ and `false` otherwise

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `A` with `A::Matrix` or `b` with `b::Vector`).
* Make sure that the function always returns either `true` or `false` irrespective of the input.

## Hints
* `basic_col_indices` need not correspond to a linearly independent set of columns (i.e., the corresponding basis matrix need not be invertible). The function must return `false` in such cases.
* Floating point numbers such as `-2.22e-16` should be considered approximately equal to `0` for all practical purposes.
    - You can check if a floating point number `f` is strictly negative using an appropriate tolerance; for example, `if f < -1e-12`
    - You can check if there exists *any* single element of a vector `x` which is strictly negative by writing `if any(x .< -1e-12)`
        - Documentation of `any` can be found [here](https://docs.julialang.org/en/v1/base/collections/#Base.any-Tuple{Any})
        - The `.` preceding `<` is called "broadcasting": Julia applies the `<` operator to each element of `x`. You can read more [here](https://docs.julialang.org/en/v1/manual/arrays/#Broadcasting) if you want to know more.
* Test your function on the examples from Lecture 8 to verify if you have implemented it correctly (you can add other tests as well)

In [3]:
function is_bfs(A, b, basic_col_indices)
    m = size(A, 1)
    n = size(A, 2)
    
    # check if input is a standard form polyhedron
    if m > n || length(b) != m || rank(A) != m
        @warn("Input is not a standard form polyhedron!")
        return false
    end
    
    # remove duplicate entries
    unique!(sort!(basic_col_indices))
    
    # check if input data is consistent
    if !issubset(basic_col_indices, 1:n) || length(basic_col_indices) != m
        @warn("basic_col_indices is not a subset of [1,...,n] of cardinality m")
        return false
    end
    
    #
    x = zeros(n,1)
    B = A[: , basic_col_indices]
    if det(B) == 0
        return false
    else
        xb = inv(B) * b       
        for i in 1:m
            x[basic_col_indices[i]] = xb[i]
        end
    end
    if any(x .< -1e-12)
        return false
    else 
        return true
    end
end

is_bfs (generic function with 1 method)

In [4]:
A = [1.0 2.0 1.0 0.0;
     2.0 1.0 0.0 1.0]
b = [3.0, 3.0]
basic_col_indices = [1,2]
is_bfs(A, b, basic_col_indices)

true

# Question 3 (30 points)
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of a **standard form polyhedron**: $P = \{x \in R^n: Ax = b, x >= 0\}$
* Assume that `basic_col_indices` is an array of $m$ numbers which correspond to $m$ column indices of `A`
    - For example, if $n=5$ and $m=3$, then `basic_col_indices = [1,4,5]` would correspond to $B(1) = 1, B(2) = 4, B(3) = 5$ in the lecture notes
* Assume that `j` is an index in $\{1, 2, \ldots, n\}$ corresponding to a nonbasic variable
* The function must return two outputs: `d`, `is_d_feasible` where
    - `d` is the $n$-dimensional direction vector corresponding to the `j`th nonbasic variable
    - `is_d_feasible` is `true` if `d` is a feasible direction and `false` otherwise

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `A` with `A::Matrix`, `b` with `b::Vector`, `j` with `j::Int` etc).
* Make sure that the function always returns its outputs in the order specified above. Interchanging the order will make it impossible to automatically score your submission on Gradescope.

## Hints
* `basic_col_indices` need not correspond to a basic feasible solution. The function must return `zeros(n), false` in such cases. You may use your function from Question 2 to check if this is the case.
* Floating point numbers such as `-2.22e-16` should be considered approximately equal to `0` for all practical purposes.
    - You can check if a floating point number `f` is strictly negative using an appropriate tolerance; for example, `if f < -1e-12`
    - You can check if a floating point number `f` is approximately equal to `0` by writing `if isapprox(f, 0.0; atol=1e-12)`
* Test your function on the examples from Lecture 12 to verify if you have implemented it correctly (you can add other tests as well)

In [5]:
function compute_basic_direction(A, b, basic_col_indices, j)
    m = size(A, 1)
    n = size(A, 2)
    
    # output vector
    d = zeros(n)
    
    # check if input is a standard form polyhedron
    if m > n || length(b) != m || rank(A) != m
        @warn("Input is not a standard form polyhedron!")
        return d, false
    end
    
    # remove duplicate entries
    unique!(sort!(basic_col_indices))
    
    # check if input data is consistent
    if !issubset(basic_col_indices, 1:n) || length(basic_col_indices) != m || j in basic_col_indices
        @warn("Input data is inconsistent!")
        return d, false
    end
    
    #
    if !is_bfs(A, b, basic_col_indices)
        @warn("Basic columns indices does not correspond to a basic feasible solution")
        return d, false
    else
        B = A[: , basic_col_indices]
        db = -1 * inv(B) * A[:,j]       
        for i in 1:m
            d[basic_col_indices[i]] = db[i]
        end
        d[j]=1
    end
    
    # calculating bfs corresponding to basic_col_indices
    x = zeros(n)
    xb = inv(B) * b       
        for i in 1:m
            x[basic_col_indices[i]] = xb[i]
        end
    
    # checking feasibility of direction with theta=0.000001
    x_new = zeros(n)
    
    for i in 1:n
        x_new[i] = x[i] + 1e-10*d[i]
    end 
    
    if any(x_new .< -1e-12)
        is_d_feasible = false
    else 
        is_d_feasible = true
    end
    
    return d, is_d_feasible
    
end

compute_basic_direction (generic function with 1 method)

In [6]:
A = [1.0 2.0 1.0 0.0;
     2.0 1.0 0.0 1.0]
b = [3.0, 3.0]
basic_col_indices = [3,4]
j = 1
compute_basic_direction(A, b, basic_col_indices, j)

([1.0, 0.0, -1.0, -2.0], true)

In [7]:
A = [1.0 2.0 1.0 0.0 0.0;
     2.0 1.0 0.0 1.0 0.0;
     2.0 0.0 0.0 0.0 1.0]
b = [3.0, 3.0, 3.0]
basic_col_indices = [1,3,4]
j = 2
compute_basic_direction(A, b, basic_col_indices, j)

([0.0, 1.0, -2.0, -1.0, 0.0], false)

# Question 4 (15 points)
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of a **standard form polyhedron**: $P = \{x \in R^n: Ax = b, x >= 0\}$
* Assume that `c` is an arbitrary $n$-dimensional cost vector
* Assume that `basic_col_indices` is an array of $m$ numbers which correspond to $m$ column indices of `A`
    - For example, if $n=5$ and $m=3$, then `basic_col_indices = [1,4,5]` would correspond to $B(1) = 1, B(2) = 4, B(3) = 5$ in the lecture notes
* The function must return the $n$-dimensional vector of reduced costs corresponding to `basic_col_indices`

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `A` with `A::Matrix`, `b` with `b::Vector`, `c` with `c::Vector` etc.).
* Make sure that the function always returns an $n$-dimensional vector irrespective of the input.

## Hints
* `basic_col_indices` need not correspond to a linearly independent set of columns (i.e., the corresponding basis matrix need not be invertible). The function must return `zeros(n)` in such cases. You may use your function from Question 2 to check if this is the case.
* Test your function on the examples from Lecture 13 to verify if you have implemented it correctly (you can add other tests as well)

In [9]:
function compute_reduced_costs(A, b, c, basic_col_indices)
    m = size(A, 1)
    n = size(A, 2)
    
    # check if input is a standard form polyhedron
    if m > n || length(b) != m || length(c) != n || rank(A) != m
        @warn("Input is not a standard form polyhedron!")
        return zeros(n)
    end
    
    # remove duplicate entries
    unique!(sort!(basic_col_indices))
    
    # check if input data is consistent
    if !issubset(basic_col_indices, 1:n) || length(basic_col_indices) != m
        @warn("basic_col_indices is not a subset of [1,...,n] of cardinality m")
        return zeros(n)
    end
    
    # check if basis matrix is invertible
    x = zeros(n,1)
    B = A[: , basic_col_indices]
    if det(B) == 0
        return zeros(n)
    
    # calculate basic solution associated with basic_col_indices 
    else
        xb = inv(B) * b       
        for i in 1:m
            x[basic_col_indices[i]] = xb[i]
        end
        
        # calculate cost vector
        reduced_costs_vector = zeros(n)
        reduced_costs_vector = c' - c[basic_col_indices]' * inv(B) * A
    
        return reduced_costs_vector
    end 
end 

compute_reduced_costs (generic function with 1 method)

In [10]:
A = [1.0 2.0 1.0 0.0;
     2.0 1.0 0.0 1.0]
b = [3.0, 3.0]
c = [-1.0, 0.0, 0.0, 0.0]
basic_col_indices = [1,3]
compute_reduced_costs(A, b, c, basic_col_indices)

1×4 adjoint(::Vector{Float64}) with eltype Float64:
 0.0  0.5  0.0  0.5

# Question 5 (40 points extra credit)
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of a **standard form polyhedron**: $P = \{x \in R^n: Ax = b, x >= 0\}$
* The function must return two non-negative integers: `num_bfs`, `num_deg_bfs`
    - The first number `num_bfs` should be the number of distinct basic feasible solutions of $P$
    - The second number `num_deg_bfs` should be the number of distinct degenerate basic feasible solutions of $P$

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `A` with `A::Matrix` or `b` with `b::Vector`).
* Make sure that the function always returns two integers in the order specified above. Interchanging the order will make it impossible to automatically score your submission on Gradescope.
* You will need to install the `Combinatorics` package for this question. There are two ways to install this package:
    1. Via the `Julia` prompt; see Section 1.2 of the `IE505-Julia-Getting-Started.pdf` document on Canvas.
    2. Alternatively, run the following in a Jupyter code cell:
        ```julia
        import Pkg
        Pkg.add("Combinatorics")
        ```
* Irrespective of which method you choose, please note that you only have to do it ONCE and after that, the package will be permanently installed on your computer. In particular, if you follow option 2, then please comment out the code cell after the package has been installed before submitting your `.ipynb` file on Gradescope.


## Hints
* The function must return the number of **distinct** basic feasible solutions - **don't double-count any solutions**!
    - This can be achieved by maintaining a list of solutions as you construct them
    - You should then add a new solution to the list only if it is not already present in the list
* You may use any of the functions from Questions 1 to 4 inside this function
* You can check if there exists *any* single element of a vector `x` approximately equal to `0` by writing `if any(isapprox.(x, 0.0; atol=1e-12))`
* You can check if two vectors `x` and `y` are approximately equal using `isapprox(x, y)`

In [109]:
#import Pkg
#Pkg.add("Combinatorics")

   Resolving package versions...
  No Changes to `C:\Users\anag_\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\anag_\.julia\environments\v1.8\Manifest.toml`


In [14]:
using Combinatorics

function count_bfs(A, b)
    m = size(A, 1)
    n = size(A, 2)
    
    # check if input is a standard form polyhedron
    if m > n || length(b) != m || rank(A) != m
        @warn("Input is not a standard form polyhedron!")
        return 0, 0
    end
    
    # list of distinct bfs
    bfs_list = []
    
    # number of bfs and degenerate bfs
    num_bfs = 0
    num_deg_bfs = 0
    M_num_bfs = zeros(n,100000) #empty matrix
    M_num_deg_bfs = zeros(n,10000)
    ii = 0
    iii = 0 #random counter

    #
    # loop over all subsets of [1, 2, .., n] of size m
    # each subset represents the indices of m columns of A
    # which will be the candidate set of basic columns
    #
    for basic_col_indices in collect(combinations(1:n, m))
        ii=ii+1
        # candidate basis matrix
        B = A[:, basic_col_indices]
        
        # check if basic_col_indices correspond to bfs
        if is_bfs(A, b, basic_col_indices)
        
            # candidate basic solution
            x = zeros(n)
            xb = inv(B) * b       
            
            for i in 1:m
                x[basic_col_indices[i]] = xb[i]
            end
            # tengo vector x
            M_num_bfs[:,ii] = x
            
            for i in x
                if i ≈ 0 
                    iii = iii+1  #cantidad de ceros
                else
                end
            end
            
            if iii > n-m
                M_num_deg_bfs[:,ii] = x
            else
            end
        else
        end      
        
    end
    
    M_num_bfs = reduce(hcat, unique(eachcol(M_num_bfs)))
    M_num_deg_bfs = reduce(hcat, unique(eachcol(M_num_deg_bfs)))
        
    num_bfs = size(M_num_bfs,2)-1
    num_deg_bfs = size(M_num_deg_bfs,2)-1
        
    return num_bfs, num_deg_bfs
end             

count_bfs (generic function with 1 method)

In [15]:
A = [1.0 2.0 1.0 0.0;
     2.0 1.0 0.0 1.0]
b = [3.0, 3.0]
count_bfs(A, b)

(4, 3)